# Explore here

In [111]:
import pandas as pd

total_data = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/regularized-linear-regression-project-tutorial/main/demographic_health_data.csv", sep = ",")
total_data.head()

,fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,...,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code
0,1001,55601,6787,12.206615,7637,13.735364,6878,12.370281,7089,12.749771,...,3644,12.9,11.9,13.8,5462,3.1,2.9,3.3,1326,3
1,1003,218022,24757,11.355276,26913,12.344167,23579,10.814964,25213,11.564429,...,14692,12.0,11.0,13.1,20520,3.2,3.0,3.5,5479,4
2,1005,24881,2732,10.980266,2960,11.896628,3268,13.134520,3201,12.865239,...,2373,19.7,18.6,20.6,3870,4.5,4.2,4.8,887,6
3,1007,22400,2456,10.964286,2596,11.589286,3029,13.522321,3113,13.897321,...,1789,14.1,13.2,14.9,2511,3.3,3.1,3.6,595,2
4,1009,57840,7095,12.266598,7570,13.087828,6742,11.656293,6884,11.901798,...,4661,13.5,12.6,14.5,6017,3.4,3.2,3.7,1507,2


In [112]:
total_data = total_data.drop_duplicates().reset_index(drop = True)
total_data.head()

,fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,...,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code
0,1001,55601,6787,12.206615,7637,13.735364,6878,12.370281,7089,12.749771,...,3644,12.9,11.9,13.8,5462,3.1,2.9,3.3,1326,3
1,1003,218022,24757,11.355276,26913,12.344167,23579,10.814964,25213,11.564429,...,14692,12.0,11.0,13.1,20520,3.2,3.0,3.5,5479,4
2,1005,24881,2732,10.980266,2960,11.896628,3268,13.134520,3201,12.865239,...,2373,19.7,18.6,20.6,3870,4.5,4.2,4.8,887,6
3,1007,22400,2456,10.964286,2596,11.589286,3029,13.522321,3113,13.897321,...,1789,14.1,13.2,14.9,2511,3.3,3.1,3.6,595,2
4,1009,57840,7095,12.266598,7570,13.087828,6742,11.656293,6884,11.901798,...,4661,13.5,12.6,14.5,6017,3.4,3.2,3.7,1507,2


Factorización 

In [113]:
import pandas as pd
import json
import os

# 1. Cargar el dataset
df = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/regularized-linear-regression-project-tutorial/main/demographic_health_data.csv")
df = df.drop_duplicates().reset_index(drop=True)

# 2. Crear carpeta para guardar los archivos JSON (opcional)
os.makedirs("factor_mappings", exist_ok=True)

# 3. Factorizar columnas tipo "object" y guardar su mapeo en JSON
for col in df.select_dtypes(include="object").columns:
    new_col = f"{col}_n"
    df[new_col] = pd.factorize(df[col])[0]

    # Crear diccionario de transformación
    mapping_dict = {
        row[col]: row[new_col]
        for _, row in df[[col, new_col]].drop_duplicates().iterrows()
    }

    # Guardar el diccionario como JSON
    filename = f"factor_mappings/{col}_transformation_rules.json"
    with open(filename, "w") as f:
        json.dump(mapping_dict, f, indent=2)

print("✅ Factorizaciones y archivos JSON guardados correctamente.")


✅ Factorizaciones y archivos JSON guardados correctamente.


Diccionario de OUTLIERS

In [114]:
import pandas as pd
import json

# 1. Cargar el dataset desde la fuente online
url = "https://raw.githubusercontent.com/4GeeksAcademy/regularized-linear-regression-project-tutorial/main/demographic_health_data.csv"
clean_data = pd.read_csv(url)
clean_data = clean_data.drop_duplicates().reset_index(drop=True)

# 2. Crear copias para trabajar con y sin outliers
clean_data_con_outliers = clean_data.copy()
clean_data_sin_outliers = clean_data.copy()

# 3. Función para reemplazar outliers por IQR
def replace_outliers_from_column(column, df):
    stats = df[column].describe()
    iqr = stats["75%"] - stats["25%"]
    upper = stats["75%"] + 1.5 * iqr
    lower = max(0, stats["25%"] - 1.5 * iqr)
    
    df[column] = df[column].apply(lambda x: min(max(x, lower), upper))
    return df.copy(), [round(lower, 2), round(upper, 2)]

# 4. Columnas numéricas (excluyendo la target si es necesario)
numeric_cols = clean_data.select_dtypes(include=["float64", "int64"]).columns.tolist()
numeric_cols.remove("Heart disease_number")  # opcional si quieres mantenerla sin tocar

# 5. Aplicar la función a cada columna y guardar los límites
outliers_dict = {}
for column in numeric_cols:
    clean_data_sin_outliers, limits_list = replace_outliers_from_column(column, clean_data_sin_outliers)
    outliers_dict[column] = limits_list

# 6. Guardar el diccionario en JSON
with open("outliers_replacement_rules.json", "w") as f:
    json.dump(outliers_dict, f, indent=2)

print("✅ Outliers reemplazados y límites guardados en 'outliers_replacement_rules.json'")


✅ Outliers reemplazados y límites guardados en 'outliers_replacement_rules.json'


GUARDAR EL DICCIONARIO DE OUTLIERS:

In [115]:
import json

# Asegúrate de que `outliers_dict` ya esté creado antes de este bloque

with open("outliers_replacement.json", "w") as f:
    json.dump(outliers_dict, f, indent=2)

# Mostrar el contenido del diccionario (opcional)
outliers_dict


{'fips': [0, np.float64(85433.0)],
 'TOT_POP': [0, np.float64(153337.62)],
 '0-9': [0, np.float64(18321.75)],
 '0-9 y/o % of total pop': [np.float64(7.06), np.float64(16.49)],
 '19-Oct': [0, np.float64(19993.88)],
 '10-19 y/o % of total pop': [np.float64(8.7), np.float64(16.64)],
 '20-29': [0, np.float64(20545.0)],
 '20-29 y/o % of total pop': [np.float64(6.47), np.float64(17.21)],
 '30-39': [0, np.float64(18936.5)],
 '30-39 y/o % of total pop': [np.float64(7.76), np.float64(15.56)],
 '40-49': [0, np.float64(18433.38)],
 '40-49 y/o % of total pop': [np.float64(8.13), np.float64(14.94)],
 '50-59': [0, np.float64(20668.62)],
 '50-59 y/o % of total pop': [np.float64(10.2), np.float64(16.96)],
 '60-69': [0, np.float64(19561.5)],
 '60-69 y/o % of total pop': [np.float64(7.35), np.float64(18.74)],
 '70-79': [0, np.float64(12508.88)],
 '70-79 y/o % of total pop': [np.float64(3.41), np.float64(13.07)],
 '80+': [0, np.float64(6449.88)],
 '80+ y/o % of total pop': [np.float64(1.12), np.float64(8

ESCALADO DE VALORES

In [116]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Crear la carpeta donde se guardarán los archivos
os.makedirs("data/processed", exist_ok=True)

# Dataset sin outliers
df = clean_data_sin_outliers.copy()

# Variables numéricas (menos la variable objetivo)
num_variables = df.select_dtypes(include=["float64", "int64"]).columns.tolist()
num_variables.remove("Heart disease_number")

# Separar X e y
X = df[num_variables]
y = df["Heart disease_number"]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalar
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=num_variables)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=num_variables)

# Agregar la variable objetivo
X_train_scaled["Heart disease_number"] = y_train.values
X_test_scaled["Heart disease_number"] = y_test.values

# Guardar archivos escalados
X_train_scaled.to_csv("data/processed/clean_train_scaled.csv", index=False)
X_test_scaled.to_csv("data/processed/clean_test_scaled.csv", index=False)

print("✅ Archivos escalados guardados correctamente en 'data/processed/'")


✅ Archivos escalados guardados correctamente en 'data/processed/'


In [117]:
%whos


Variable                       Type                          Data/Info
----------------------------------------------------------------------
Lasso                          ABCMeta                       <class 'sklearn.linear_mo<...>oordinate_descent.Lasso'>
LogisticRegression             type                          <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
SelectKBest                    ABCMeta                       <class 'sklearn.feature_s<...>e_selection.SelectKBest'>
StandardScaler                 type                          <class 'sklearn.preproces<...>ng._data.StandardScaler'>
X                              DataFrame                            fips     TOT_POP  <...>[3140 rows x 105 columns]
X_sin                          DataFrame                            fips     TOT_POP  <...>[3140 rows x 105 columns]
X_test                         DataFrame                            fips     TOT_POP  <...>n[628 rows x 105 columns]
X_test_con_norm                DataFram

In [118]:
import os

# Ruta destino
output_dir = "/Users/mariannacastro/Desktop/4Geeks/proyectos/Build-a-linear-regression-modek/data/processed"
os.makedirs(output_dir, exist_ok=True)

# Guardar datasets escalados
X_train_scaled.to_excel(f"{output_dir}/X_train_scaled.xlsx", index=False)
X_test_scaled.to_excel(f"{output_dir}/X_test_scaled.xlsx", index=False)
y_train.to_excel(f"{output_dir}/y_train.xlsx", index=False)
y_test.to_excel(f"{output_dir}/y_test.xlsx", index=False)

print("✅ Datasets escalados guardados correctamente en Excel.")


✅ Datasets escalados guardados correctamente en Excel.


In [119]:
X_train.head()


,fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,...,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code
1292,26127,26625.0,3221.0,12.097653,3463.0,13.006573,2922.0,10.974648,2829.0,10.625352,...,2314.0,13.7,12.6,14.9,2823.0,3.8,3.5,4.1,771.0,6
2302,42121,51266.0,5272.0,10.283619,5751.0,11.217961,5137.0,10.020286,5341.0,10.418211,...,4097.0,13.1,11.9,14.2,5416.0,3.5,3.2,3.8,1454.0,5
761,18133,37779.0,3915.0,10.362900,5118.0,13.547209,6202.0,16.416528,4363.0,11.548744,...,2792.0,12.2,11.2,13.1,3698.0,2.9,2.7,3.1,871.0,2
2194,40131,91984.0,11163.0,12.135806,12646.0,13.748043,11595.0,12.605453,11357.0,12.346712,...,5716.0,11.2,10.4,12.0,7913.0,3.0,2.8,3.2,2118.0,3
1241,26025,134487.0,16698.0,12.416070,17666.0,13.135842,17281.0,12.849569,15993.0,11.891856,...,10002.0,12.5,11.7,13.4,12987.0,3.4,3.2,3.6,3490.0,4


NORMALIZACIÓN

In [120]:
import os
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Crear carpeta para guardar normalizadores
normalizer_dir = "models/normalizers"
os.makedirs(normalizer_dir, exist_ok=True)

# 2. Dataset con outliers (ya tienes X_train)
scaler_con_outliers = StandardScaler()
scaler_con_outliers.fit(X_train)

# Guardar el normalizador con outliers
with open(os.path.join(normalizer_dir, "normalizador_con_outliers.pkl"), "wb") as file:
    pickle.dump(scaler_con_outliers, file)

# 3. Dataset sin outliers (necesitamos crearlo)
# Obtener variables numéricas del dataset sin outliers
num_vars = clean_data_sin_outliers.select_dtypes(include=["float64", "int64"]).columns.tolist()
num_vars.remove("Heart disease_number")

# Dividir el dataset sin outliers
X_sin = clean_data_sin_outliers[num_vars]
y_sin = clean_data_sin_outliers["Heart disease_number"]
X_train_sin, _, _, _ = train_test_split(X_sin, y_sin, test_size=0.2, random_state=42)

# 4. Normalizador sin outliers
scaler_sin_outliers = StandardScaler()
scaler_sin_outliers.fit(X_train_sin)

# Guardar el normalizador sin outliers
with open(os.path.join(normalizer_dir, "normalizador_sin_outliers.pkl"), "wb") as file:
    pickle.dump(scaler_sin_outliers, file)

print("✅ Normalizadores guardados en 'models/normalizers/' correctamente.")


✅ Normalizadores guardados en 'models/normalizers/' correctamente.


Crear los datasets normalizados con y sin outliers y guardarlos.

In [121]:
import pandas as pd
import os
import pickle
from sklearn.model_selection import train_test_split

# Crear carpeta si no existe
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

# Cargar los normalizadores previamente guardados
with open("models/normalizers/normalizador_con_outliers.pkl", "rb") as f:
    scaler_con = pickle.load(f)

with open("models/normalizers/normalizador_sin_outliers.pkl", "rb") as f:
    scaler_sin = pickle.load(f)

# ----------------------------
# Dataset CON outliers
# ----------------------------

# Ya tienes:
# - X_train, X_test
# - y_train, y_test

X_train_con_norm = pd.DataFrame(scaler_con.transform(X_train), columns=X_train.columns)
X_test_con_norm = pd.DataFrame(scaler_con.transform(X_test), columns=X_test.columns)
X_train_con_norm["Heart disease_number"] = y_train.values
X_test_con_norm["Heart disease_number"] = y_test.values

# Guardar
X_train_con_norm.to_excel(f"{output_dir}/X_train_con_outliers_norm.xlsx", index=False)
X_test_con_norm.to_excel(f"{output_dir}/X_test_con_outliers_norm.xlsx", index=False)

# ----------------------------
# Dataset SIN outliers
# ----------------------------

# Volver a generar X_sin e y_sin
num_vars = clean_data_sin_outliers.select_dtypes(include=["float64", "int64"]).columns.tolist()
num_vars.remove("Heart disease_number")

X_sin = clean_data_sin_outliers[num_vars]
y_sin = clean_data_sin_outliers["Heart disease_number"]

X_train_sin, X_test_sin, y_train_sin, y_test_sin = train_test_split(
    X_sin, y_sin, test_size=0.2, random_state=42
)

X_train_sin_norm = pd.DataFrame(scaler_sin.transform(X_train_sin), columns=X_train_sin.columns)
X_test_sin_norm = pd.DataFrame(scaler_sin.transform(X_test_sin), columns=X_test_sin.columns)
X_train_sin_norm["Heart disease_number"] = y_train_sin.values
X_test_sin_norm["Heart disease_number"] = y_test_sin.values

# Guardar
X_train_sin_norm.to_excel(f"{output_dir}/X_train_sin_outliers_norm.xlsx", index=False)
X_test_sin_norm.to_excel(f"{output_dir}/X_test_sin_outliers_norm.xlsx", index=False)

print("✅ Datasets normalizados (con y sin outliers) guardados como archivos Excel en 'data/processed/'")


✅ Datasets normalizados (con y sin outliers) guardados como archivos Excel en 'data/processed/'


ESCALADO MÍNIMO-MÁXIMO

In [122]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Crear carpetas si no existen
output_dir = "data/processed"
scaler_dir = "models/scalers"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(scaler_dir, exist_ok=True)

# ----------------------------
# CON OUTLIERS
# ----------------------------
# X_train y X_test ya existen
scaler_con = MinMaxScaler()
X_train_con_scal = pd.DataFrame(scaler_con.fit_transform(X_train), columns=X_train.columns)
X_test_con_scal = pd.DataFrame(scaler_con.transform(X_test), columns=X_test.columns)

X_train_con_scal["Heart disease_number"] = y_train.values
X_test_con_scal["Heart disease_number"] = y_test.values

# Guardar los datasets
X_train_con_scal.to_excel(f"{output_dir}/X_train_con_outliers_scal.xlsx", index=False)
X_test_con_scal.to_excel(f"{output_dir}/X_test_con_outliers_scal.xlsx", index=False)

# Guardar el scaler
with open(f"{scaler_dir}/scaler_con_outliers.pkl", "wb") as f:
    pickle.dump(scaler_con, f)

# ----------------------------
# SIN OUTLIERS
# ----------------------------
# Reconstruir X_train_sin y X_test_sin
num_vars = clean_data_sin_outliers.select_dtypes(include=["float64", "int64"]).columns.tolist()
num_vars.remove("Heart disease_number")

X_sin = clean_data_sin_outliers[num_vars]
y_sin = clean_data_sin_outliers["Heart disease_number"]

X_train_sin, X_test_sin, y_train_sin, y_test_sin = train_test_split(
    X_sin, y_sin, test_size=0.2, random_state=42
)

scaler_sin = MinMaxScaler()
X_train_sin_scal = pd.DataFrame(scaler_sin.fit_transform(X_train_sin), columns=X_train_sin.columns)
X_test_sin_scal = pd.DataFrame(scaler_sin.transform(X_test_sin), columns=X_test_sin.columns)

X_train_sin_scal["Heart disease_number"] = y_train_sin.values
X_test_sin_scal["Heart disease_number"] = y_test_sin.values

# Guardar los datasets
X_train_sin_scal.to_excel(f"{output_dir}/X_train_sin_outliers_scal.xlsx", index=False)
X_test_sin_scal.to_excel(f"{output_dir}/X_test_sin_outliers_scal.xlsx", index=False)

# Guardar el scaler
with open(f"{scaler_dir}/scaler_sin_outliers.pkl", "wb") as f:
    pickle.dump(scaler_sin, f)

print("✅ Datasets escalados (MinMax) y scalers guardados correctamente.")


✅ Datasets escalados (MinMax) y scalers guardados correctamente.


In [123]:
from sklearn.preprocessing import StandardScaler

data_types = total_data.dtypes
numeric_columns = [c for c in list(data_types[data_types != "object"].index) if c != "Heart disease_number"]

scaler = StandardScaler()
norm_features = scaler.fit_transform(total_data[numeric_columns])

# Create a new DataFrame with the scaled numerical variables
total_data_scal = pd.DataFrame(norm_features, index = total_data.index, columns = numeric_columns)
total_data_scal["Heart disease_number"] = total_data["Heart disease_number"]
total_data_scal.head()

,fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,...,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code,Heart disease_number
0,-1.940874,-0.145679,-0.142421,0.158006,-0.135556,0.573496,-0.153144,0.027610,-0.139384,0.588469,...,-0.063696,-0.071720,-0.089834,-0.129902,-0.609615,-0.582796,-0.669652,-0.147523,-1.082865,3345
1,-1.940742,0.341296,0.287476,-0.242861,0.320383,-0.193107,0.183774,-0.469965,0.230620,-0.110300,...,-0.394103,-0.414900,-0.337677,0.376251,-0.433549,-0.393279,-0.343373,0.389791,-0.420704,13414
2,-1.940610,-0.237785,-0.239429,-0.419441,-0.246181,-0.439718,-0.225971,0.272104,-0.218759,0.656538,...,2.432709,2.483064,2.317776,-0.183415,1.855312,1.880929,1.777443,-0.204321,0.903618,2159
3,-1.940478,-0.245223,-0.246032,-0.426966,-0.254791,-0.609076,-0.230792,0.396168,-0.220555,1.264959,...,0.376846,0.423984,0.299632,-0.229096,-0.257483,-0.203761,-0.180233,-0.242100,-1.745026,1533
4,-1.940346,-0.138966,-0.135053,0.186249,-0.137140,0.216679,-0.155888,-0.200808,-0.143570,0.088582,...,0.156575,0.195197,0.158008,-0.111247,-0.081417,-0.014244,-0.017093,-0.124105,-1.745026,4101


In [124]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression

X = total_data_scal.drop(columns=["Heart disease_number"])
y = total_data_scal["Heart disease_number"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
train_indices = list(X_train.index)
test_indices = list(X_test.index)

k = int(len(X_train.columns) * 0.3)
selection_model = SelectKBest(score_func = f_regression, k = k)
selection_model.fit(X_train, y_train)
ix = selection_model.get_support()

X_train_sel = pd.DataFrame(selection_model.transform(X_train), columns = X_train.columns.values[ix])
X_test_sel = pd.DataFrame(selection_model.transform(X_test), columns = X_test.columns.values[ix])

X_train_sel.head()

,TOT_POP,0-9,19-Oct,20-29,30-39,40-49,50-59,60-69,70-79,80+,...,Family Medicine/General Practice Primary Care (2019),Total Specialist Physicians (2019),Total Population,Population Aged 60+,county_pop2018_18 and older,anycondition_number,Obesity_number,COPD_number,diabetes_number,CKD_number
0,-0.232556,-0.227731,-0.234284,-0.232951,-0.226353,-0.231316,-0.229599,-0.233425,-0.234680,-0.234420,...,-0.212643,-0.208590,-0.231195,-0.229737,-0.233171,-0.234370,-0.232975,-0.223516,-0.218609,-0.219329
1,-0.158676,-0.178665,-0.180166,-0.188266,-0.175070,-0.161168,-0.134688,-0.105618,-0.119270,-0.091822,...,-0.116680,-0.110850,-0.150293,-0.098866,-0.152859,-0.142645,-0.155304,-0.110080,-0.131449,-0.130962
2,-0.199114,-0.211128,-0.195138,-0.166782,-0.195036,-0.194045,-0.199725,-0.219256,-0.222207,-0.205154,...,-0.192263,-0.217668,-0.197005,-0.216056,-0.195125,-0.193205,-0.201976,-0.193106,-0.189197,-0.206391
3,-0.036595,-0.037734,-0.017077,-0.057986,-0.052252,-0.033158,-0.020228,-0.032603,-0.023876,-0.046224,...,0.062458,-0.107888,-0.036940,-0.030034,-0.039882,-0.003321,0.006163,-0.007077,-0.047515,-0.045054
4,0.090839,0.094680,0.101662,0.056721,0.042392,0.068095,0.101699,0.144664,0.140685,0.166099,...,0.274818,0.194913,0.097767,0.161314,0.088485,0.165555,0.182740,0.265603,0.123040,0.132454


In [125]:
X_test_sel.head()


,TOT_POP,0-9,19-Oct,20-29,30-39,40-49,50-59,60-69,70-79,80+,...,Family Medicine/General Practice Primary Care (2019),Total Specialist Physicians (2019),Total Population,Population Aged 60+,county_pop2018_18 and older,anycondition_number,Obesity_number,COPD_number,diabetes_number,CKD_number
0,-0.285286,-0.285362,-0.294836,-0.269566,-0.258568,-0.268541,-0.289649,-0.312989,-0.316763,-0.286734,...,-0.303292,-0.285225,-0.284324,-0.308211,-0.283698,-0.302439,-0.302292,-0.324038,-0.276290,-0.281172
1,0.496553,0.433072,0.392170,0.544659,0.453677,0.391480,0.499744,0.668639,0.716353,0.476084,...,0.853184,0.424904,0.477184,0.620724,0.517408,0.527360,0.516364,0.443806,0.418504,0.454092
2,-0.260191,-0.255123,-0.265837,-0.246628,-0.234723,-0.240703,-0.264552,-0.289867,-0.289846,-0.290962,...,-0.277451,-0.261868,-0.257294,-0.287868,-0.259943,-0.249299,-0.259877,-0.225107,-0.225970,-0.242229
3,0.039389,0.058341,0.059701,-0.018647,0.003236,0.030594,0.074401,0.091003,0.060721,0.005012,...,0.197267,0.130719,0.036299,0.055281,0.031494,0.035274,0.026108,0.136643,0.003409,0.022352
4,0.364272,0.281232,0.323623,0.525353,0.295990,0.288317,0.298029,0.461297,0.497760,0.351393,...,0.659217,0.305024,0.336581,0.423969,0.390596,0.271127,0.273318,0.329669,0.256620,0.334804


In [126]:
X_train_sel["Heart disease_number"] = list(y_train)
X_test_sel["Heart disease_number"] = list(y_test)

X_train_sel.to_csv("../data/processed/clean_train.csv", index = False)
X_test_sel.to_csv("../data/processed/clean_test.csv", index = False)

In [127]:
total_data = pd.concat([X_train_sel, X_test_sel])
total_data.head()

,TOT_POP,0-9,19-Oct,20-29,30-39,40-49,50-59,60-69,70-79,80+,...,Total Specialist Physicians (2019),Total Population,Population Aged 60+,county_pop2018_18 and older,anycondition_number,Obesity_number,COPD_number,diabetes_number,CKD_number,Heart disease_number
0,-0.232556,-0.227731,-0.234284,-0.232951,-0.226353,-0.231316,-0.229599,-0.233425,-0.234680,-0.234420,...,-0.208590,-0.231195,-0.229737,-0.233171,-0.234370,-0.232975,-0.223516,-0.218609,-0.219329,2072
1,-0.158676,-0.178665,-0.180166,-0.188266,-0.175070,-0.161168,-0.134688,-0.105618,-0.119270,-0.091822,...,-0.110850,-0.150293,-0.098866,-0.152859,-0.142645,-0.155304,-0.110080,-0.131449,-0.130962,3796
2,-0.199114,-0.211128,-0.195138,-0.166782,-0.195036,-0.194045,-0.199725,-0.219256,-0.222207,-0.205154,...,-0.217668,-0.197005,-0.216056,-0.195125,-0.193205,-0.201976,-0.193106,-0.189197,-0.206391,2222
3,-0.036595,-0.037734,-0.017077,-0.057986,-0.052252,-0.033158,-0.020228,-0.032603,-0.023876,-0.046224,...,-0.107888,-0.036940,-0.030034,-0.039882,-0.003321,0.006163,-0.007077,-0.047515,-0.045054,5484
4,0.090839,0.094680,0.101662,0.056721,0.042392,0.068095,0.101699,0.144664,0.140685,0.166099,...,0.194913,0.097767,0.161314,0.088485,0.165555,0.182740,0.265603,0.123040,0.132454,8686


In [128]:
X_test_sel.head()


,TOT_POP,0-9,19-Oct,20-29,30-39,40-49,50-59,60-69,70-79,80+,...,Total Specialist Physicians (2019),Total Population,Population Aged 60+,county_pop2018_18 and older,anycondition_number,Obesity_number,COPD_number,diabetes_number,CKD_number,Heart disease_number
0,-0.285286,-0.285362,-0.294836,-0.269566,-0.258568,-0.268541,-0.289649,-0.312989,-0.316763,-0.286734,...,-0.285225,-0.284324,-0.308211,-0.283698,-0.302439,-0.302292,-0.324038,-0.276290,-0.281172,698
1,0.496553,0.433072,0.392170,0.544659,0.453677,0.391480,0.499744,0.668639,0.716353,0.476084,...,0.424904,0.477184,0.620724,0.517408,0.527360,0.516364,0.443806,0.418504,0.454092,13982
2,-0.260191,-0.255123,-0.265837,-0.246628,-0.234723,-0.240703,-0.264552,-0.289867,-0.289846,-0.290962,...,-0.261868,-0.257294,-0.287868,-0.259943,-0.249299,-0.259877,-0.225107,-0.225970,-0.242229,1768
3,0.039389,0.058341,0.059701,-0.018647,0.003236,0.030594,0.074401,0.091003,0.060721,0.005012,...,0.130719,0.036299,0.055281,0.031494,0.035274,0.026108,0.136643,0.003409,0.022352,6739
4,0.364272,0.281232,0.323623,0.525353,0.295990,0.288317,0.298029,0.461297,0.497760,0.351393,...,0.305024,0.336581,0.423969,0.390596,0.271127,0.273318,0.329669,0.256620,0.334804,11305


Modelo de regresión logística

In [129]:
train_data = pd.read_csv("../data/processed/clean_train.csv")
test_data = pd.read_csv("../data/processed/clean_test.csv")

train_data.head()

,TOT_POP,0-9,19-Oct,20-29,30-39,40-49,50-59,60-69,70-79,80+,...,Total Specialist Physicians (2019),Total Population,Population Aged 60+,county_pop2018_18 and older,anycondition_number,Obesity_number,COPD_number,diabetes_number,CKD_number,Heart disease_number
0,-0.232556,-0.227731,-0.234284,-0.232951,-0.226353,-0.231316,-0.229599,-0.233425,-0.234680,-0.234420,...,-0.208590,-0.231195,-0.229737,-0.233171,-0.234370,-0.232975,-0.223516,-0.218609,-0.219329,2072
1,-0.158676,-0.178665,-0.180166,-0.188266,-0.175070,-0.161168,-0.134688,-0.105618,-0.119270,-0.091822,...,-0.110850,-0.150293,-0.098866,-0.152859,-0.142645,-0.155304,-0.110080,-0.131449,-0.130962,3796
2,-0.199114,-0.211128,-0.195138,-0.166782,-0.195036,-0.194045,-0.199725,-0.219256,-0.222207,-0.205154,...,-0.217668,-0.197005,-0.216056,-0.195125,-0.193205,-0.201976,-0.193106,-0.189197,-0.206391,2222
3,-0.036595,-0.037734,-0.017077,-0.057986,-0.052252,-0.033158,-0.020228,-0.032603,-0.023876,-0.046224,...,-0.107888,-0.036940,-0.030034,-0.039882,-0.003321,0.006163,-0.007077,-0.047515,-0.045054,5484
4,0.090839,0.094680,0.101662,0.056721,0.042392,0.068095,0.101699,0.144664,0.140685,0.166099,...,0.194913,0.097767,0.161314,0.088485,0.165555,0.182740,0.265603,0.123040,0.132454,8686


In [130]:
X_train = train_data.drop(["Heart disease_number"], axis = 1)
y_train = train_data["Heart disease_number"]
X_test = test_data.drop(["Heart disease_number"], axis = 1)
y_test = test_data["Heart disease_number"]

In [131]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [132]:
print(f"Intercep (a): {model.intercept_}")
print(f"Coefficients: {model.coef_}")

Intercep (a): [-0.31577655 -0.31611833 -0.31450123 ... -2.23736428 -2.20380037
 -2.1309372 ]
Coefficients: [[-0.0713727  -0.06668874 -0.07305192 ... -0.11869662 -0.0828258
  -0.08349982]
 [-0.07151184 -0.06733433 -0.07368701 ... -0.11869433 -0.08269389
  -0.0831639 ]
 [-0.07121635 -0.06669565 -0.07307112 ... -0.11843195 -0.08256185
  -0.08319106]
 ...
 [ 0.2165989   0.2683158   0.31337754 ...  0.2749301  -0.09471925
   0.31334123]
 [ 0.19361095  0.14203426  0.09838972 ...  0.23859774  0.11540521
   0.3023436 ]
 [ 0.22885701  0.09555643  0.12295956 ...  0.02857736  0.42052461
   0.28869374]]


In [133]:
y_pred = model.predict(X_test)
y_pred

array([ 1072,  8689,  1072,  8689,  7128,  1072,  1072,  1072,  1072,
        1072,  1072,  1072,  7128,  1072,  7128,  1072, 75432,  1072,
        1072,  1072,  1072,  1072,  1072,  1072, 40686,  1072,  1072,
        1072,  1072,  1072,  1072,  1072,  1072,  1072,  7128,  1072,
        1072,  1072,  1072,  1072,  1072,  7128,  1072,  1072,  1072,
        1072,  1072,  8506,  1072,  1072, 32863,  1072,  1072,  1072,
        8506,  1072, 31550,  1072,  1072,  1072,  7128,  1072,  1072,
        8506,  1072,  1072,  1072,  1072,  1072, 38899,  1072,  1072,
        1072,  1072,  1072,  1448, 32863,  1072,  1072,  1072,  1072,
        3376, 12367,  1072,  1072,  1072, 32828,  1072,  1072,  7128,
        1072,  1072,  1072,  1072,  1072,  1072,  1072,  1072,  8506,
        1072, 76128,  8689,  1072,  1072,  1072,  1072,  1072,  1072,
        7128, 23631,  1072,  7011, 25091,  3376,  1072, 16376,  1072,
        1072,  7128,  1072,  1072,  1072,  1072,  1072,  1072,  1072,
        3376,  1072,

In [134]:
from sklearn.metrics import mean_squared_error, r2_score

print(f"MSE: {mean_squared_error(y_test, y_pred)}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")

MSE: 24126640.40286624
R2 Score: 0.724347227415247


Optimización del modelo

In [135]:
from sklearn.linear_model import Lasso

alpha = 1.0
lasso_model = Lasso(alpha = alpha)

# Training the model
lasso_model.fit(X_train, y_train)

# We evaluate the performance of the model on the test data
score = lasso_model.score(X_test, y_test)
print("Coefficients:", lasso_model.coef_)
print("R2 score:", score)

Coefficients: [ 5103.56606854  1192.14607797 -1921.90787729  -804.66413704
  -565.56094295  4161.43524651   552.93901319 -1080.72356488
  3459.52199626  1245.55139019   999.98373671 -5424.05510818
   198.01474247  -841.91968637  -371.17714777  2792.66368052
   324.15462356    19.27250203 -1918.35380595    88.70731834
   492.63537754  -461.24851762  -854.99744188 -2893.16049233
  3359.53043536   204.58782867  1925.46994753  2907.60993035
  5383.37174712   819.64992462 -2329.43640877]
R2 score: 0.9978911816625889


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.505e+08, tolerance: 7.097e+07
  model = cd_fast.enet_coordinate_descent(


In [136]:
from pickle import dump

dump(lasso_model, open("../models/lasso_alpha-1.0.sav", "wb"))